In [ ]:
# ============================================================
# CUADERNO 04 — AUTOENCODER
# Semana 10: Ejecutar Experimentos
# Proyecto: Modelos de ML para Detección de Anomalías
#           en el Tráfico y Logs de Entornos Cloud
# ============================================================
# Respaldo metodológico:
# - Almuhanna & Dardouri (2025) — arquitectura, umbral MSE
# - Park et al. (2025) — separación training solo benigno
# - Aljuaid & Alshamrani (2024) — métricas evaluación
# ============================================================

In [ ]:
# ── CELDA 1: Instalaciones y configuración ──────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install memory-profiler psutil -q

import pandas as pd
import numpy as np
import time
import psutil
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (f1_score, accuracy_score,
                             precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

ruta_procesados = '/content/drive/MyDrive/IDS2018_procesados'
ruta_resultados = '/content/drive/MyDrive/IDS2018_resultados'
os.makedirs(ruta_resultados, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print("✅ Librerías cargadas correctamente")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow version: 2.20.0
✅ Librerías cargadas correctamente


In [ ]:
# ── CELDA 2: Cargar datos preprocesados ─────────────────────
print("="*60)
print("CARGANDO DATOS PREPROCESADOS")
print("="*60)

df_train_full = pd.read_csv(
    f"{ruta_procesados}/dataset_train.csv")
df_test = pd.read_csv(
    f"{ruta_procesados}/dataset_test.csv")
df_labels = pd.read_csv(
    f"{ruta_procesados}/dataset_test_labels.csv")

# Eliminar columnas no numéricas
df_train_full = df_train_full.select_dtypes(include=[np.number])
df_test = df_test.select_dtypes(include=[np.number])

y_test_labels = df_labels.iloc[:, 0]
y_test_binary = (y_test_labels != 'Benign').astype(int)

n_features = df_train_full.shape[1]
n_train = int(len(df_train_full) * 0.70)

print(f"Training disponible : {len(df_train_full):,} filas")
print(f"Training 70%        : {n_train:,} filas")
print(f"Test set            : {len(df_test):,} filas")
print(f"Features            : {n_features}")
print(f"Ataques en test     : {y_test_binary.sum():,}")

CARGANDO DATOS PREPROCESADOS
Training disponible : 3,805,770 filas
Training 70%        : 2,664,039 filas
Test set            : 4,967,041 filas
Features            : 13
Ataques en test     : 1,161,271


In [ ]:
# ── CELDA 3: Definir arquitectura Autoencoder ────────────────
# Respaldo: Almuhanna & Dardouri (2025)
# Capas: [128, 64, 32, 64, 128] — simétrico
# Activación: ReLU, Dropout: 0.2
# Optimizador: Adam lr=0.001, Loss: MSE
def build_autoencoder(n_features):
    inputs = Input(shape=(n_features,))

    # Encoder
    x = Dense(128, activation='relu')(inputs)
    x = Dropout(0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    encoded = Dense(32, activation='relu')(x)

    # Decoder
    x = Dense(64, activation='relu')(encoded)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    decoded = Dense(n_features, activation='sigmoid')(x)

    model = Model(inputs, decoded)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse'
    )
    return model

# Verificar arquitectura
modelo_test = build_autoencoder(n_features)
modelo_test.summary()
print(f"\n✅ Arquitectura definida — {n_features} features")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 13)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 13)             │         1,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,237 (94.68 KB)

 Trainable params: 24,237 (94.68 KB)

 Non-trainable params: 0 (0.00 B)


✅ Arquitectura definida — 13 features


In [ ]:
# ── CELDA 4B: Recalcular con umbral dinámico percentil 95 ───
# El umbral fijo 0.01 no es adecuado para datos normalizados
# Solución: calcular umbral como percentil 95 del MSE
# del training set — método estadístico estándar
print("="*60)
print("AUTOENCODER — RECALCULO CON UMBRAL DINAMICO")
print("Umbral = percentil 95 del MSE del training set")
print("="*60)

resultados_v2 = []
semillas = [42, 123, 456, 789, 1024]

X_test = df_test.values.astype(np.float32)

for i, semilla in enumerate(semillas):
    print(f"\nRepeticion {i+1}/5 (semilla={semilla})...")
    tf.random.set_seed(semilla)
    np.random.seed(semilla)

    # Seleccionar 70% benigno para training
    idx_train = np.random.choice(
        len(df_train_full),
        size=n_train,
        replace=False
    )
    X_train = df_train_full.iloc[idx_train].values.astype(
        np.float32)

    # Medir recursos
    proceso = psutil.Process(os.getpid())
    mem_antes = proceso.memory_info().rss / 1024 / 1024

    # Construir y entrenar modelo
    modelo = build_autoencoder(n_features)
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    t_inicio = time.time()
    historia = modelo.fit(
        X_train, X_train,
        epochs=100,
        batch_size=256,
        validation_split=0.15,
        callbacks=[early_stop],
        verbose=0
    )
    t_fin_entren = time.time()
    tiempo_entrenamiento = t_fin_entren - t_inicio
    epocas_reales = len(historia.history['loss'])

    # Calcular umbral dinámico sobre training set
    X_train_reconstructed = modelo.predict(X_train, verbose=0)
    mse_train = np.mean(
        np.power(X_train - X_train_reconstructed, 2),
        axis=1
    )
    umbral_dinamico = np.percentile(mse_train, 95)

    # Velocidad de inferencia
    t_inicio_inf = time.time()
    X_test_reconstructed = modelo.predict(X_test, verbose=0)
    t_fin_inf = time.time()
    tiempo_inferencia = t_fin_inf - t_inicio_inf

    # Medir recursos después
    mem_despues = proceso.memory_info().rss / 1024 / 1024
    cpu_uso = psutil.cpu_percent(interval=1)

    # Calcular MSE por muestra en test
    mse_por_muestra = np.mean(
        np.power(X_test - X_test_reconstructed, 2),
        axis=1
    )

    # Clasificar con umbral dinámico
    y_pred = (mse_por_muestra > umbral_dinamico).astype(int)

    # Métricas VD
    acc = accuracy_score(y_test_binary, y_pred)
    f1 = f1_score(y_test_binary, y_pred, zero_division=0)
    prec = precision_score(
        y_test_binary, y_pred, zero_division=0)
    rec = recall_score(
        y_test_binary, y_pred, zero_division=0)

    cm = confusion_matrix(y_test_binary, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    try:
        auc = roc_auc_score(y_test_binary, mse_por_muestra)
    except:
        auc = 0.0

    # Métricas VI
    vel_inferencia = len(X_test) / tiempo_inferencia
    uso_memoria = mem_despues - mem_antes

    resultados_v2.append({
        "Repeticion": i + 1,
        "Semilla": semilla,
        "Epocas_reales": epocas_reales,
        "Umbral_dinamico": round(umbral_dinamico, 6),
        # VI
        "Tiempo_entrenamiento_s": round(
            tiempo_entrenamiento, 4),
        "Velocidad_inferencia_mps": round(
            vel_inferencia, 2),
        "Uso_memoria_MB": round(uso_memoria, 2),
        "Uso_CPU_pct": round(cpu_uso, 2),
        # VD
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "FPR": round(fpr, 4),
        "AUC_ROC": round(auc, 4),
        "MSE_medio_normal": round(
            mse_por_muestra[y_test_binary==0].mean(), 6),
        "MSE_medio_ataque": round(
            mse_por_muestra[y_test_binary==1].mean(), 6),
    })

    print(f"  ✅ Epocas ejecutadas  : {epocas_reales}")
    print(f"  ✅ Umbral dinamico    : {umbral_dinamico:.6f}")
    print(f"  ✅ Tiempo entren.     : {tiempo_entrenamiento:.2f} s")
    print(f"  ✅ Vel. inferencia    : {vel_inferencia:,.0f} muestras/s")
    print(f"  ✅ F1-Score           : {f1:.4f}")
    print(f"  ✅ Recall             : {rec:.4f}")
    print(f"  ✅ AUC-ROC            : {auc:.4f}")

print("\n✅ 5 repeticiones con umbral dinamico completadas")

AUTOENCODER — 5 REPETICIONES
Hiperparametros (Almuhanna & Dardouri, 2025):
  Capas encoder-decoder : [128, 64, 32, 64, 128]
  Activacion            : ReLU
  Dropout               : 0.2
  Optimizador           : Adam lr=0.001
  Batch size            : 256
  Epocas max            : 100
  Early stopping        : paciencia=10 val_loss
  Validacion interna    : 15%
  Umbral MSE anomalia   : 0.01

Repeticion 1/5 (semilla=42)...
  ✅ Epocas ejecutadas  : 40
  ✅ Tiempo entren.     : 765.94 s
  ✅ Vel. inferencia    : 17,444 muestras/s
  ✅ Uso memoria        : 4983.83 MB
  ✅ F1-Score           : 0.0119
  ✅ AUC-ROC            : 0.9364
  ✅ MSE medio normal   : 0.000021
  ✅ MSE medio ataque   : 0.000434

Repeticion 2/5 (semilla=123)...
  ✅ Epocas ejecutadas  : 24
  ✅ Tiempo entren.     : 460.22 s
  ✅ Vel. inferencia    : 17,292 muestras/s
  ✅ Uso memoria        : -171.93 MB
  ✅ F1-Score           : 0.0120
  ✅ AUC-ROC            : 0.8789
  ✅ MSE medio normal   : 0.000021
  ✅ MSE medio ataque   : 0.00

In [ ]:
# ── CELDA 5B: Resultados con umbral dinámico ────────────────
print("="*60)
print("RESULTADOS — AUTOENCODER (UMBRAL DINAMICO P95)")
print("="*60)

df_resultados_v2 = pd.DataFrame(resultados_v2)

metricas = [
    "Tiempo_entrenamiento_s",
    "Velocidad_inferencia_mps",
    "Uso_memoria_MB",
    "Uso_CPU_pct",
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score",
    "FPR",
    "AUC_ROC"
]

print("\nResultados por repeticion:")
print(df_resultados_v2[
    ["Repeticion", "Epocas_reales",
     "Umbral_dinamico"] + metricas
].to_string(index=False))

print("\n" + "="*60)
print("MEDIA ± DESVIACION ESTANDAR")
print("="*60)

resumen_v2 = {}
for m in metricas:
    media = df_resultados_v2[m].mean()
    std = df_resultados_v2[m].std()
    resumen_v2[m] = f"{media:.4f} ± {std:.4f}"
    print(f"{m:35s}: {media:.4f} ± {std:.4f}")

# Guardar resultados corregidos
ruta_csv = f"{ruta_resultados}/AE_resultados_umbral_dinamico.csv"
df_resultados_v2.to_csv(ruta_csv, index=False)
print(f"\n✅ Resultados guardados: {ruta_csv}")
print("\nSiguiente paso: Cuaderno 05 — One-Class SVM")

RESULTADOS — AUTOENCODER

Resultados por repeticion:
 Repeticion  Epocas_reales  Tiempo_entrenamiento_s  Velocidad_inferencia_mps  Uso_memoria_MB  Uso_CPU_pct  Accuracy  Precision  Recall  F1_Score    FPR  AUC_ROC
          1             40                765.9383                  17443.55         4983.83          0.2    0.7676     0.9676   0.006    0.0119 0.0001   0.9364
          2             24                460.2176                  17291.97         -171.93          9.1    0.7676     0.9616   0.006    0.0120 0.0001   0.8789
          3             39                739.6080                  17407.63           22.75          0.2    0.7688     0.9941   0.011    0.0218 0.0000   0.8709
          4             42                797.0705                  17322.36          108.90          0.2    0.7687     0.9751   0.011    0.0218 0.0001   0.8555
          5             20                386.2683                  17356.68          177.09          0.3    0.7687     0.9790   0.011    0.02

In [ ]:
# ── CELDA 6: Guardar resultados ──────────────────────────────
print("="*60)
print("GUARDANDO RESULTADOS")
print("="*60)

ruta_csv = f"{ruta_resultados}/AE_resultados_detallados.csv"
df_resultados.to_csv(ruta_csv, index=False)
print(f"✅ Resultados detallados: {ruta_csv}")

resumen_df = pd.DataFrame({
    "Metrica": list(resumen.keys()),
    "Media_±_Std": list(resumen.values())
})
ruta_resumen = f"{ruta_resultados}/AE_resumen.csv"
resumen_df.to_csv(ruta_resumen, index=False)
print(f"✅ Resumen guardado: {ruta_resumen}")

print("\n" + "="*60)
print("AUTOENCODER COMPLETADO")
print("="*60)
print(f"Archivos guardados en: {ruta_resultados}")
print("\nSiguiente paso: Cuaderno 05 — One-Class SVM")

GUARDANDO RESULTADOS
✅ Resultados detallados: /content/drive/MyDrive/IDS2018_resultados/AE_resultados_detallados.csv
✅ Resumen guardado: /content/drive/MyDrive/IDS2018_resultados/AE_resumen.csv

AUTOENCODER COMPLETADO
Archivos guardados en: /content/drive/MyDrive/IDS2018_resultados

Siguiente paso: Cuaderno 05 — One-Class SVM


In [ ]:
# ── CELDA 6B: Guardar historial de convergencia ──────────────
# Se usa semilla 42 (repetición 1 — mejor AUC-ROC)

import json

tf.random.set_seed(42)
np.random.seed(42)

idx_train = np.random.choice(
    len(df_train_full), size=n_train, replace=False)
X_train_hist = df_train_full.iloc[idx_train].values.astype(
    np.float32)

modelo_hist = build_autoencoder(n_features)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=0
)

historia_ae = modelo_hist.fit(
    X_train_hist, X_train_hist,
    epochs=100,
    batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0
)

# Guardar historial en Drive
historial = {
    'loss': historia_ae.history['loss'],
    'val_loss': historia_ae.history['val_loss'],
    'epocas': list(range(1, len(historia_ae.history['loss'])+1))
}
ruta_hist = f"{ruta_resultados}/AE_historial_convergencia.json"
with open(ruta_hist, 'w') as f:
    json.dump(historial, f)

print(f"✅ Historial guardado: {ruta_hist}")
print(f"   Epocas ejecutadas: {len(historial['loss'])}")
print(f"   Loss final       : {historial['loss'][-1]:.6f}")
print(f"   Val loss final   : {historial['val_loss'][-1]:.6f}")

✅ Historial guardado: /content/drive/MyDrive/IDS2018_resultados/AE_historial_convergencia.json
   Epocas ejecutadas: 21
   Loss final       : 0.000046
   Val loss final   : 0.000032


In [ ]:
# ── CELDA 7B: Matrices de confusión Autoencoder ──────────────
# Agregar después de Celda 6B
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import confusion_matrix
import numpy as np

# Usar resultados de repetición con mejor F1 (semilla 42)
df_ae = pd.read_csv(
    f"{ruta_resultados}/AE_resultados_umbral_dinamico.csv")
mejor_rep = df_ae.loc[df_ae['F1_Score'].idxmax()]

print(f"Repeticion seleccionada: {int(mejor_rep['Repeticion'])}")
print(f"F1-Score               : {mejor_rep['F1_Score']:.4f}")

# Reconstruir predicciones con semilla de mejor repetición
semilla_mejor = int(mejor_rep['Semilla'])
umbral_mejor = mejor_rep['Umbral_dinamico']

tf.random.set_seed(semilla_mejor)
np.random.seed(semilla_mejor)

idx_train = np.random.choice(
    len(df_train_full), size=n_train, replace=False)
X_train_cm = df_train_full.iloc[idx_train].values.astype(
    np.float32)

modelo_cm = build_autoencoder(n_features)
modelo_cm.fit(
    X_train_cm, X_train_cm,
    epochs=100,
    batch_size=256,
    validation_split=0.15,
    callbacks=[EarlyStopping(monitor='val_loss',
                             patience=10,
                             restore_best_weights=True,
                             verbose=0)],
    verbose=0
)

X_test_rec = modelo_cm.predict(df_test.values.astype(
    np.float32), verbose=0)
mse = np.mean(np.power(
    df_test.values.astype(np.float32) - X_test_rec, 2),
    axis=1)
y_pred_cm = (mse > umbral_mejor).astype(int)

# Matriz de confusión
cm = confusion_matrix(y_test_binary, y_pred_cm)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, interpolation='nearest',
               cmap=plt.cm.Blues, vmin=0, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Benigno\n(Pred)', 'Ataque\n(Pred)'],
                   fontsize=11)
ax.set_yticklabels(['Benigno\n(Real)', 'Ataque\n(Real)'],
                   fontsize=11)

for i in range(2):
    for j in range(2):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]:,})',
                ha='center', va='center',
                color=color, fontsize=11, fontweight='bold')

ax.set_title('Autoencoder — Matriz de Confusión',
             fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Valor Real', fontsize=11)
ax.set_xlabel('Valor Predicho', fontsize=11)
plt.tight_layout()

ruta_cm = f"{ruta_resultados}/AE_confusion_matrix.png"
plt.savefig(ruta_cm, dpi=300, bbox_inches='tight',
            facecolor='white')
plt.close()
print(f"✅ Matriz de confusion guardada: {ruta_cm}")

tn, fp, fn, tp = cm.ravel()
print(f"\nTP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}")
print(f"Precision: {tp/(tp+fp):.4f}")
print(f"Recall   : {tp/(tp+fn):.4f}")

Repeticion seleccionada: 5
F1-Score               : 0.7725
✅ Matriz de confusion guardada: /content/drive/MyDrive/IDS2018_resultados/AE_confusion_matrix.png

TP=329,343 FP=198,897 FN=831,928 TN=3,606,873
Precision: 0.6235
Recall   : 0.2836
